# Enveda CASMI 2026 — complete-data audit & exploratory analysis

**Notebook version 0.6.0 · 2026-11-19 UTC · self-contained (no project `src/` imports)**

## What this notebook does
It streams **every row and every column** of `train.parquet` and `test.parquet` in Arrow batches, calculates full-dataset row/null/category/ID/array-quality counts, and keeps only a bounded, reproducible row sample for charts. This is a complete scan, not a full in-memory DataFrame. The train file is about 3 GB compressed; expanding every nested peak array in pandas can require far more RAM than the file size.

The notebook contains its own loader, validators, data dictionary, synthetic self-test and plotting code. It does not import `src` or other project files. Only standard/scientific Python libraries are installed.

> **Validation warning:** Kaggle says the visible `test.parquet` is a training-derived preview, replaced by hidden test data during notebook rerun. Never treat it as independent validation. This notebook performs EDA only; it does not train a model or claim a leaderboard score.

## 0. Dependencies and file locations

For Colab, mount Drive and set `DATA_DIR` below. For local Jupyter, place both Parquet files in a folder and set the same path. This notebook requires permission to read the files; it does not handle or store Kaggle credentials.

In [ ]:
%pip -q install numpy pandas pyarrow matplotlib seaborn

from pathlib import Path
import json, math, random, re, sqlite3, tempfile, time
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Optional Colab Drive mount; uncomment when the files are stored in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = Path('./data')  # change to e.g. Path('/content/drive/MyDrive/casmi_data')
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
OUTPUT_DIR = Path('./casmi_eda_v0.6.0')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 20_000       # lower this if Arrow batch memory is tight
PLOT_SAMPLE_ROWS = 12_000 # reservoir sample for plots only; full data is still scanned
SEED = 42
for path in (TRAIN_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Missing required competition file: {path}')
print({p.name: f'{p.stat().st_size / (1024**3):.2f} GiB on disk' for p in (TRAIN_PATH, TEST_PATH)})


## 1. Official data dictionary (guide; validate actual Arrow schema)

| Field(s) | Meaning / why it matters |
|---|---|
| `molecule_id`, `spectrum_id` | Prediction unit versus individual spectrum. Multiple spectra per molecule are expected; `spectrum_id` should be unique. |
| `ms2_mzs`, `ms2_normalized_intensities` | Aligned fragment m/z and abundance arrays. Index `i` in both arrays describes one peak; normalized base peak should be near 1. |
| `precursor_mz`, `adduct`, `ionization_mode` | Parent-ion measurement and chemistry needed to estimate neutral mass. Adduct mistakes contaminate formula/candidate retrieval. |
| `collision_energy_ev`, `collision_energy_orig`, `collision_energy_orig_units` | Fragmentation condition; source units vary in train. `collision_energy_ev` is a best-effort conversion. |
| `base_peak_intensity`, `num_peaks`, `precursor_error_ppm` | Signal/quality and consistency checks; some fields are train-only or nullable by source. |
| `instrument_type`, `ingest_lib`, `adduct_orig` | Domain/source metadata; train is heterogeneous, test is documented as timsTOF. |
| `normalized_smiles`, `inchikey`, `inchikey14`, `molecular_formula` | Training labels and chemical identity; never read as test labels. |

The Kaggle page enumerates ten test adduct forms but also says “one of the seven above”; it labels test 12 columns while its train-column heading/count appears inconsistent with the listed additional fields. The actual Parquet schema reported below is authoritative.

In [ ]:
# Parse numeric values from energy metadata, including lists and strings such as '35HCD'.
def flatten_numeric(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, np.ndarray)):
        return [number for item in value for number in flatten_numeric(item)]
    if isinstance(value, (int, float, np.number)):
        return [float(value)] if np.isfinite(value) else []
    if isinstance(value, str):
        return [float(token) for token in re.findall(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?', value)]
    return []


# Validate a single paired peak-list record and return counts useful for the full-file audit.
def inspect_peak_arrays(mzs, intensities):
    result = {'peak_count': 0, 'pair_length_error': 0, 'invalid_mz_values': 0, 'invalid_intensity_values': 0, 'negative_intensities': 0, 'intensity_over_one': 0, 'base_peak_not_one': 0}
    try:
        mz_array = np.asarray(mzs, dtype=np.float64)
        intensity_array = np.asarray(intensities, dtype=np.float64)
    except (TypeError, ValueError):
        result['pair_length_error'] = 1
        return result
    if mz_array.ndim != 1 or intensity_array.ndim != 1 or len(mz_array) != len(intensity_array):
        result['pair_length_error'] = 1
        return result
    result['peak_count'] = len(mz_array)
    result['invalid_mz_values'] = int((~np.isfinite(mz_array) | (mz_array < 0)).sum())
    result['invalid_intensity_values'] = int((~np.isfinite(intensity_array)).sum())
    finite_intensity = intensity_array[np.isfinite(intensity_array)]
    result['negative_intensities'] = int((finite_intensity < 0).sum())
    result['intensity_over_one'] = int((finite_intensity > 1.001).sum())
    positive = finite_intensity[finite_intensity > 0]
    if len(positive) and not np.isclose(positive.max(), 1.0, atol=0.01):
        result['base_peak_not_one'] = 1
    return result


# Scan every Parquet row in bounded batches; compute exact counts and retain a uniform reservoir sample.
def scan_complete_parquet(path, label, sample_capacity=12_000, batch_size=20_000, seed=42):
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    metadata_rows = parquet.metadata.num_rows
    column_names = schema.names
    null_counts = Counter()
    categories = {name: Counter() for name in ('ingest_lib', 'adduct', 'ionization_mode', 'instrument_type', 'collision_energy_orig_units')}
    molecule_spectra = Counter()
    molecule_ids = set()
    peak_audit = Counter({name: 0 for name in ('peak_count', 'pair_length_error', 'invalid_mz_values', 'invalid_intensity_values', 'negative_intensities', 'intensity_over_one', 'base_peak_not_one', 'num_peaks_metadata_mismatch_rows', 'num_peaks_metadata_invalid_rows')})
    spectrum_id_nonnull_rows = 0
    precursor_min, precursor_max = float('inf'), float('-inf')
    sample_records, seen_rows, rows = [], 0, 0
    rng = random.Random(seed)
    with tempfile.TemporaryDirectory(prefix=f'{label}_spectrum_ids_') as scratch:
        db_path = Path(scratch) / 'ids.sqlite'
        conn = sqlite3.connect(db_path)
        conn.execute('PRAGMA journal_mode=OFF')
        conn.execute('PRAGMA synchronous=OFF')
        conn.execute('CREATE TABLE spectrum_ids (id TEXT PRIMARY KEY) WITHOUT ROWID')
        for batch in parquet.iter_batches(batch_size=batch_size, columns=column_names, use_threads=True):
            frame = batch.to_pandas()
            rows += len(frame)
            for col in frame.columns:
                null_counts[col] += int(frame[col].isna().sum())
            for col, counts in categories.items():
                if col in frame.columns:
                    counts.update(frame[col].fillna('<NULL>').astype(str).value_counts().to_dict())
            if 'precursor_mz' in frame.columns:
                precursor = pd.to_numeric(frame['precursor_mz'], errors='coerce').to_numpy(dtype=float)
                finite = precursor[np.isfinite(precursor)]
                if len(finite):
                    precursor_min = min(precursor_min, float(finite.min()))
                    precursor_max = max(precursor_max, float(finite.max()))
            records = frame.to_dict(orient='records')
            batch_spectrum_ids = []
            for record in records:
                molecule_id = record.get('molecule_id')
                if molecule_id is not None and not pd.isna(molecule_id):
                    molecule_id = str(molecule_id)
                    molecule_ids.add(molecule_id)
                    molecule_spectra[molecule_id] += 1
                spectrum_id = record.get('spectrum_id')
                if spectrum_id is not None and not pd.isna(spectrum_id):
                    batch_spectrum_ids.append((str(spectrum_id),))
                    spectrum_id_nonnull_rows += 1
                if 'ms2_mzs' in record and 'ms2_normalized_intensities' in record:
                    peak_audit.update(inspect_peak_arrays(record['ms2_mzs'], record['ms2_normalized_intensities']))
                    if 'num_peaks' in record and pd.notna(record['num_peaks']):
                        try:
                            if int(record['num_peaks']) != len(record['ms2_mzs']):
                                peak_audit['num_peaks_metadata_mismatch_rows'] += 1
                        except (TypeError, ValueError):
                            peak_audit['num_peaks_metadata_invalid_rows'] += 1
                seen_rows += 1
                if len(sample_records) < sample_capacity:
                    sample_records.append(record)
                elif sample_capacity > 0:
                    replacement = rng.randrange(seen_rows)
                    if replacement < sample_capacity:
                        sample_records[replacement] = record
            if batch_spectrum_ids:
                conn.executemany('INSERT OR IGNORE INTO spectrum_ids(id) VALUES (?)', batch_spectrum_ids)
            conn.commit()
            del records, frame, batch_spectrum_ids
        unique_spectrum_ids = conn.execute('SELECT COUNT(*) FROM spectrum_ids').fetchone()[0]
        conn.close()
    report = {
        'label': label, 'path': str(path), 'file_bytes': path.stat().st_size, 'parquet_metadata_rows': int(metadata_rows), 'rows_scanned': rows, 'row_count_matches_metadata': bool(rows == metadata_rows),
        'unique_molecules': len(molecule_ids),
        'spectra_per_molecule_distribution': dict(Counter(molecule_spectra.values()).most_common()),
        'spectra_per_molecule_quantiles': np.quantile(list(molecule_spectra.values()), [0, .25, .5, .75, .95, .99, 1]).tolist() if molecule_spectra else [],
        'spectrum_id_nonnull_rows': int(spectrum_id_nonnull_rows) if 'spectrum_id' in column_names else None,
        'unique_spectrum_ids': int(unique_spectrum_ids) if 'spectrum_id' in column_names else None,
        'duplicate_spectrum_id_rows': int(spectrum_id_nonnull_rows - unique_spectrum_ids) if 'spectrum_id' in column_names else None,
        'columns': [{'name': field.name, 'arrow_type': str(field.type), 'nullable': field.nullable} for field in schema],
        'null_counts': dict(null_counts), 'category_counts': {name: dict(counter.most_common()) for name, counter in categories.items() if counter},
        'peak_array_audit': dict(peak_audit),
        'precursor_mz_min_exact': precursor_min if np.isfinite(precursor_min) else None,
        'precursor_mz_max_exact': precursor_max if np.isfinite(precursor_max) else None,
        'plot_sample_rows': len(sample_records), 'quantile_note': 'precursor and per-spectrum peak quantiles are estimates from the uniform bounded plot sample; row/category/null/ID counts scan the complete file.'
    }
    return report, pd.DataFrame.from_records(sample_records)


### Why use a reservoir sample?
The scanner reads every row, not only the sample. Exact row totals, null counts, category counts, molecule/spectrum ID counts, peak-array audit counts and precursor min/max are computed across the full file. The sample is only for plots and quantiles, which avoids retaining millions of nested arrays in RAM. Spectrum-ID uniqueness is checked exactly with a temporary disk-backed SQLite index rather than a large Python set.

In [ ]:
# Self-test the notebook's scanner using a tiny synthetic Parquet file before scanning multi-GB inputs.
import tempfile
import pyarrow as pa
with tempfile.TemporaryDirectory() as temp:
    toy = pd.DataFrame({
        'molecule_id': ['a', 'a', 'b', 'b'], 'spectrum_id': ['s1', 's2', 's2', None],
        'precursor_mz': [200.0, 201.0, 300.0, 301.0], 'ms2_mzs': [[50.0, 80.0], [60.0], [70.0, 90.0], [55.0]],
        'ms2_normalized_intensities': [[1.0, 0.5], [1.0], [0.8, 0.2], [1.0]], 'adduct': ['[M+H]+'] * 4,
        'ingest_lib': ['L1', 'L1', 'L2', 'L2'], 'num_peaks': [2, 1, 2, 1]
    })
    toy_path = Path(temp) / 'toy.parquet'
    pq.write_table(pa.Table.from_pandas(toy), toy_path)
    toy_report, toy_sample = scan_complete_parquet(toy_path, 'toy', sample_capacity=2, batch_size=1)
assert toy_report['rows_scanned'] == 4
assert toy_report['row_count_matches_metadata'] is True
assert toy_report['unique_molecules'] == 2
assert toy_report['spectrum_id_nonnull_rows'] == 3
assert toy_report['duplicate_spectrum_id_rows'] == 1
assert toy_report['peak_array_audit']['pair_length_error'] == 0
assert toy_report['peak_array_audit']['num_peaks_metadata_mismatch_rows'] == 0
assert len(toy_sample) == 2
print('Synthetic scanner self-test passed:', {k: toy_report[k] for k in ('rows_scanned', 'unique_molecules', 'unique_spectrum_ids', 'duplicate_spectrum_id_rows')})


## 2. Full train/test scans

This cell processes all train and test Parquet rows in batches. It can take minutes and requires temporary disk space for the exact duplicate-spectrum-ID index. Lower `BATCH_SIZE` if RAM is tight. `PLOT_SAMPLE_ROWS` changes only the plot sample size; it does **not** drop data from the scan.

In [ ]:
started = time.time()
reports, samples = {}, {}
for label, path in (('train', TRAIN_PATH), ('test', TEST_PATH)):
    print(f'Scanning every row in {path.name} ...')
    reports[label], samples[label] = scan_complete_parquet(path, label, sample_capacity=PLOT_SAMPLE_ROWS, batch_size=BATCH_SIZE, seed=SEED)
    print({k: reports[label][k] for k in ('rows_scanned', 'unique_molecules', 'unique_spectrum_ids', 'duplicate_spectrum_id_rows', 'plot_sample_rows')})
(OUTPUT_DIR / 'full_data_audit.json').write_text(json.dumps(reports, indent=2, default=str))
print(f"Finished full scan in {(time.time() - started) / 60:.1f} minutes; report: {OUTPUT_DIR / 'full_data_audit.json'}")
display(pd.DataFrame({label: {
    'rows_scanned': report['rows_scanned'], 'unique_molecules': report['unique_molecules'],
    'unique_spectrum_ids': report['unique_spectrum_ids'], 'duplicate_spectrum_id_rows': report['duplicate_spectrum_id_rows'],
    'plot_sample_rows': report['plot_sample_rows'],
    'precursor_mz_min_exact': report['precursor_mz_min_exact'], 'precursor_mz_max_exact': report['precursor_mz_max_exact']
} for label, report in reports.items()}).T)


## 3. Full-scan data-quality findings

Use exact counts below to answer: Are spectrum IDs unique? Do arrays align? Does `num_peaks` match parsed arrays? Are intensities normalized? Are there missing precursor/adduct/label fields? Do adduct and energy conventions vary by source? The plots are descriptive; they do not establish predictive performance.

In [ ]:
for label, report in reports.items():
    print(f'===== {label.upper()} SCHEMA =====')
    display(pd.DataFrame(report['columns']))
    print(f'===== {label.upper()} NULL COUNTS (exact) =====')
    display(pd.Series(report['null_counts'], name='null_rows').sort_values(ascending=False).to_frame())
    print(f'===== {label.upper()} PEAK / ID VALIDATION (exact) =====')
    display(pd.Series(report['peak_array_audit'], name='count').to_frame())
    print(f'===== {label.upper()} CATEGORY COUNTS (exact) =====')
    display({key: pd.Series(value).head(30) for key, value in report['category_counts'].items()})


## 4. Composite visualization — each panel answers a data question

Mass and peak distributions diagnose coverage/domain shift; spectra-per-molecule verifies aggregation granularity; source/adduct plots expose heterogeneity; missingness highlights unreliable metadata; the peak diagram demonstrates the raw representation. Plot distributions use the bounded uniform sample, except exact source/adduct/missingness and group-count panels read from full-scan counters.

In [ ]:
# Convert per-row energy metadata to a flattened numeric array for visualization.
def energy_values(frame):
    if 'collision_energy_ev' not in frame.columns:
        return []
    return [number for value in frame['collision_energy_ev'] for number in flatten_numeric(value)]


# Plot full-scan counts and sampled distributions as a compact diagnostic dashboard.
def plot_composite(reports, samples, output_path):
    sns.set_theme(style='whitegrid', context='notebook')
    fig, ax = plt.subplots(3, 3, figsize=(19, 15))
    colors = {'train': '#176d70', 'test': '#d98232'}
    # A. Precursor mass: compare data domains, keeping all rows represented in sampling notes.
    for label, frame in samples.items():
        if 'precursor_mz' in frame:
            sns.histplot(pd.to_numeric(frame['precursor_mz'], errors='coerce').dropna(), bins=55, stat='density', element='step', fill=False, ax=ax[0,0], color=colors[label], label=label)
    ax[0,0].set(title='Precursor m/z — sample overlay', xlabel='precursor m/z', ylabel='density')
    ax[0,0].legend()
    # B. Peak counts are long-tailed, so the y axis is logarithmic.
    for label, frame in samples.items():
        if 'ms2_mzs' in frame:
            n = frame['ms2_mzs'].map(lambda values: len(values) if isinstance(values, (list, tuple, np.ndarray)) else 0)
            sns.histplot(n, bins=45, element='step', fill=False, ax=ax[0,1], color=colors[label], label=label)
    ax[0,1].set_yscale('log'); ax[0,1].set(title='Fragments per spectrum (sample)', xlabel='peak count', ylabel='log rows')
    ax[0,1].legend()
    # C. Number of spectra per molecule, exact full-data frequency counts.
    if 'train' in reports and reports['train']['spectra_per_molecule_distribution']:
        dist = pd.Series(reports['train']['spectra_per_molecule_distribution']).sort_index()
        ax[0,2].bar(dist.index.astype(int), dist.values, color=colors['train'])
        ax[0,2].set_yscale('log'); ax[0,2].set(title='Train spectra per molecule (exact)', xlabel='spectra per molecule', ylabel='molecules (log)')
    else:
        ax[0,2].axis('off')
    # D. Training source distribution (exact counts; top 15).
    if 'train' in reports and 'ingest_lib' in reports['train']['category_counts']:
        src = pd.Series(reports['train']['category_counts']['ingest_lib']).sort_values().tail(15)
        src.plot.barh(ax=ax[1,0], color=colors['train']); ax[1,0].set_xscale('log')
        ax[1,0].set(title='Training spectra by source (exact)', xlabel='spectra (log)', ylabel='ingest_lib')
    else:
        ax[1,0].axis('off')
    # E. Adduct profiles by dataset (exact category counts).
    adducts = sorted(set().union(*(set(r['category_counts'].get('adduct', {})) for r in reports.values())))
    adduct_table = pd.DataFrame({label: report['category_counts'].get('adduct', {}) for label, report in reports.items()}).fillna(0).reindex(adducts).fillna(0)
    if len(adduct_table):
        adduct_table.plot.bar(ax=ax[1,1], width=.85, color=[colors[k] for k in adduct_table.columns])
        ax[1,1].set_yscale('log'); ax[1,1].set(title='Adducts by dataset (exact)', xlabel='adduct', ylabel='spectra (log)'); ax[1,1].tick_params(axis='x', rotation=55)
    else:
        ax[1,1].axis('off')
    # F. Precursor m/z against fragment count; sample shows mass/complexity relationship.
    frame = samples.get('train', pd.DataFrame())
    if {'precursor_mz', 'ms2_mzs'}.issubset(frame.columns):
        count = frame['ms2_mzs'].map(lambda values: len(values) if isinstance(values, (list, tuple, np.ndarray)) else 0)
        ax[1,2].hexbin(pd.to_numeric(frame['precursor_mz'], errors='coerce'), count, gridsize=35, mincnt=1, cmap='viridis', xscale='linear')
        ax[1,2].set_yscale('log')
        ax[1,2].set(title='Train precursor vs fragment count (sample)', xlabel='precursor m/z', ylabel='peaks (log)')
    else:
        ax[1,2].axis('off')
    # G. Collision-energy distributions; values in merged lists are flattened for display.
    for label, frame in samples.items():
        energies = energy_values(frame)
        if energies:
            sns.histplot(energies, bins=35, stat='density', element='step', fill=False, ax=ax[2,0], color=colors[label], label=label)
    ax[2,0].set(title='Collision energy eV (sample)', xlabel='energy (eV)', ylabel='density')
    if ax[2,0].get_legend_handles_labels()[0]: ax[2,0].legend()
    # H. Missingness across all columns (exact full-scan null counts).
    labels, missing_fraction = [], []
    for label, report in reports.items():
        for col, n_missing in report['null_counts'].items():
            labels.append(f'{label}:{col}'); missing_fraction.append(n_missing / max(1, report['rows_scanned']))
    if labels:
        order = np.argsort(missing_fraction)[-18:]
        ax[2,1].barh(np.asarray(labels)[order], np.asarray(missing_fraction)[order], color='#8055a5')
        ax[2,1].set(title='Most-missing fields (exact)', xlabel='fraction null', xlim=(0,1))
    # I. A representative query spectrum illustrates paired m/z/intensity values.
    frame = samples.get('test', pd.DataFrame())
    if len(frame) and 'ms2_mzs' in frame and 'ms2_normalized_intensities' in frame:
        peak_counts = frame['ms2_mzs'].map(lambda values: len(values) if isinstance(values, (list, tuple, np.ndarray)) else 0)
        row = frame.iloc[int(np.argmin(np.abs(peak_counts.to_numpy() - np.median(peak_counts))))]
        try:
            mz = np.asarray(row['ms2_mzs'], dtype=float); inten = np.asarray(row['ms2_normalized_intensities'], dtype=float)
            valid = np.isfinite(mz) & np.isfinite(inten)
            ax[2,2].vlines(mz[valid], 0, inten[valid], color='#176d70', linewidth=1)
            ax[2,2].set(title=f'Representative test spectrum ({valid.sum()} peaks)', xlabel='fragment m/z', ylabel='normalized intensity', ylim=(0,1.05))
        except (TypeError, ValueError):
            ax[2,2].axis('off')
    else:
        ax[2,2].axis('off')
    fig.suptitle('CASMI 2026 — complete-file scan with bounded visualization sample', fontsize=16)
    fig.tight_layout(rect=(0,0,1,.97))
    fig.savefig(output_path, dpi=170, bbox_inches='tight')
    plt.close(fig)

plot_path = OUTPUT_DIR / 'complete_data_eda.png'
plot_composite(reports, samples, plot_path)
display(Image(filename=str(plot_path)))
print('Plot saved:', plot_path)


## 5. Focused visualizations and interpretation prompts

Use these to answer the questions with actual numbers from your downloaded files; do not invent conclusions from the published approximate counts. A domain shift in precursor/peak distributions or source mismatch is evidence to stratify validation, not proof a model will score poorly.

In [ ]:
# Compare adduct frequency within ionization mode in the bounded sample.
for label, frame in samples.items():
    if {'adduct', 'ionization_mode'}.issubset(frame.columns):
        print(label, 'adduct × ionization mode (sample row proportions)')
        display(pd.crosstab(frame['adduct'].fillna('<NULL>'), frame['ionization_mode'].fillna('<NULL>'), normalize='columns').round(3))

# Compare precursor and peak-count quantiles from the bounded uniform sample.
for label, frame in samples.items():
    summary = {}
    if 'precursor_mz' in frame:
        summary['precursor_mz'] = pd.to_numeric(frame['precursor_mz'], errors='coerce').quantile([0,.25,.5,.75,.95,1]).to_dict()
    if 'ms2_mzs' in frame:
        summary['peaks_per_spectrum'] = frame['ms2_mzs'].map(lambda values: len(values) if isinstance(values, (list, tuple, np.ndarray)) else 0).quantile([0,.25,.5,.75,.95,1]).to_dict()
    print(label, 'sampled quantiles (not full-file quantiles):')
    display(summary)


## 6. Export and next step

The complete scan report is `casmi_eda_v0.6.0/full_data_audit.json`; the plot is `casmi_eda_v0.6.0/complete_data_eda.png`. Share the JSON and your observations—not the multi-GB training file—to continue.

### Acceptance checklist
- [ ] Both Parquet files scanned to completion (`rows_scanned` matches Parquet metadata / Kaggle documentation scale).
- [ ] Actual Arrow schema reviewed; documentation contradictions resolved from local schema.
- [ ] Duplicate spectrum IDs, duplicate/missing molecule IDs, null counts and peak-array integrity reviewed.
- [ ] Spectra-per-molecule distribution and source/adduct/instrument/energy coverage reviewed.
- [ ] Composite graphs and sample quantiles interpreted; no model conclusions drawn from the preview.
- [ ] Audit JSON and plot saved and retained for the next iteration.

This notebook completes the *data-audit implementation* only. It does not resolve Tasks 3–8 (retrieval, candidate recall, learning, ranking, or offline submission). Those require evidence from actual files and leakage-safe validation.